# Redrob Hackathon — Candidate Ranking Demo Sandbox

**Team:** Mysterio

This notebook demonstrates the full ranking pipeline on a small candidate sample (≤100 candidates).
It does NOT require Google Drive access or any external API calls.

## Instructions
1. Run all cells top to bottom (Runtime → Run all)
2. When prompted in Cell 3, upload a small `candidates.jsonl` sample (first 100 lines of the full file)
3. The pipeline will rank them and produce a CSV

**Compute budget:** Runs in under 5 minutes on CPU. No GPU required for ranking.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q sentence-transformers pandas pyarrow scikit-learn tqdm

In [ ]:
# Cell 2 — Create folder structure and clone scripts from GitHub
import os

os.makedirs('/content/redrob/data', exist_ok=True)
os.makedirs('/content/redrob/artifacts', exist_ok=True)
os.makedirs('/content/redrob/outputs', exist_ok=True)
os.makedirs('/content/redrob/scripts', exist_ok=True)

# Clone scripts directly from GitHub — no manual upload needed
!git clone https://github.com/aditya-singh2005/Redrob-Hackathon-AI-Recruitment-Engine /content/redrob-repo
import shutil
for f in ['01_parse_jd.py', '02_feature_gen.py', '03_rank.py']:
    shutil.copy(f'/content/redrob-repo/scripts/{f}', f'/content/redrob/scripts/{f}')
print('Scripts loaded from GitHub ✓')

In [ ]:
# Cell 3 — Upload a small candidate sample (first 100 lines of candidates.jsonl)
# To create the sample on your machine:
#   Windows PowerShell: Get-Content data\candidates.jsonl | Select-Object -First 100 | Out-File -Encoding utf8 sample_candidates_100.jsonl
#   Mac/Linux:          head -100 data/candidates.jsonl > sample_candidates_100.jsonl

from google.colab import files
print('Upload your sample_candidates_100.jsonl file:')
uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, '/content/redrob/data/candidates.jsonl')
    lines = sum(1 for _ in open('/content/redrob/data/candidates.jsonl'))
    print(f'Uploaded {fname} → {lines} candidates')

In [ ]:
# Cell 4 — Run script 01: parse JD into structured requirements
%cd /content/redrob
!python scripts/01_parse_jd.py

In [ ]:
# Cell 5 — Run script 02: extract features + embed profiles
# On this small sample (~100 candidates) this runs in under 60 seconds on CPU
!python scripts/02_feature_gen.py

In [ ]:
# Cell 6 — Run script 03: rank candidates (CPU only, no API calls)
!python scripts/03_rank.py --out outputs/sandbox_output.csv

In [ ]:
# Cell 7 — View results and download
import pandas as pd
from google.colab import files

df = pd.read_csv('/content/redrob/outputs/sandbox_output.csv')
print(f'Ranked {len(df)} candidates')
print(df[['candidate_id','rank','score','reasoning']].to_string())


In [ ]:
# Cell 8 — Download the ranked CSV
# Run this cell separately after Cell 7 finishes printing
from google.colab import files
files.download('/content/redrob/outputs/sandbox_output.csv')
print('Download triggered — check your browser downloads ✓')